In [0]:
%pip install databricks-ai-search --upgrade mlflow[databricks] ragas langchain-community langchain-google-vertexai
dbutils.library.restartPython()

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-auth[pyopenssl] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 150.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 159.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Not uninstalling pydantic-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephem

In [0]:
from mlflow.deployments import get_deploy_client
from databricks.ai_search.client import VectorSearchClient

In [0]:
dbutils.widgets.text("user_question", "What are the main customer concerns about product delivery?")
dbutils.widgets.text("llm_endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("vs_endpoint_name", "sales-endpoint-rag_agentic")
dbutils.widgets.text("customer_feedback_index", "rag_agentic.workday_demos.customer_feedback_index")
dbutils.widgets.text("meeting_notes_index", "rag_agentic.workday_demos.meeting_notes_index")
dbutils.widgets.text("email_communications_index", "rag_agentic.workday_demos.email_communications_index")

user_question = dbutils.widgets.get("user_question")
llm_endpoint_name = dbutils.widgets.get("llm_endpoint_name")
vs_endpoint_name = dbutils.widgets.get("vs_endpoint_name")
customer_feedback_index = dbutils.widgets.get("customer_feedback_index")
meeting_notes_index = dbutils.widgets.get("meeting_notes_index")
email_communications_index = dbutils.widgets.get("email_communications_index")

# Define your indexes
indexes = {
            'customer_feedback': customer_feedback_index,
            'meeting_notes': meeting_notes_index,
            'email_communications': email_communications_index
        }

In [0]:
def generate_rag_prompt(user_question: str, 
                        customer_feedback_context: str,
                        meeting_notes_context: str, 
                        email_context: str) -> str:
    """
    Generate a RAG prompt with context from three knowledge sources
    """
    
    prompt = f"""You are an intelligent assistant with access to information from customer feedback, meeting notes, and email communications.

    ## Retrieved Context:

    ### Customer Feedback:
    {customer_feedback_context}

    ### Meeting Notes:
    {meeting_notes_context}

    ### Email Communications:
    {email_context}

    ## User Question:
    {user_question}

    ## Instructions:
    1. Analyze the context from all three sources carefully
    2. Synthesize information across customer feedback, meeting notes, and emails to provide a comprehensive answer
    3. If information conflicts between sources, note the discrepancy and provide context
    4. If the context doesn't contain relevant information, acknowledge this limitation
    5. Cite which source(s) your answer comes from (customer feedback, meeting notes, or emails)
    6. Provide specific examples or quotes when relevant

    ## Answer:
    """
    return prompt


def format_results(results):
    """Format search results into readable context"""
    if not results or 'result' not in results or 'data_array' not in results['result']:
        return "No relevant information found."
    
    formatted = []
    for i, row in enumerate(results['result']['data_array'], 1):
        content = row[0] if len(row) > 0 else "N/A"
        doc_uri = row[1] if len(row) > 1 else "Unknown source"
        formatted.append(f"[{i}] Source: {doc_uri}\n{content}\n")
    
    return "\n".join(formatted)


def retrieve_contexts(vsc, vs_endpoint_name, indexes, user_question: str, num_results: int = 5):
    """
    Retrieve relevant context from all three indexes
    """
    contexts = {}
    
    # Query customer feedback index
    feedback_index = vsc.get_index(
                                    endpoint_name=vs_endpoint_name,
                                    index_name=indexes['customer_feedback']
                                )
    feedback_results = feedback_index.similarity_search(
                                                        query_text=user_question,
                                                        columns=["content", "doc_uri"],
                                                        num_results=num_results
                                                    )
    contexts['customer_feedback'] = format_results(feedback_results)
    
    # Query meeting notes index
    meeting_index = vsc.get_index(
                                    endpoint_name='sales-endpoint-rag_agentic',
                                    index_name=indexes['meeting_notes']
                                )
    meeting_results = meeting_index.similarity_search(
                                                        query_text=user_question,
                                                        columns=["content", "doc_uri"],
                                                        num_results=num_results
                                                    )
    contexts['meeting_notes'] = format_results(meeting_results)
    
    # Query email communications index
    email_index = vsc.get_index(
                                endpoint_name='sales-endpoint-rag_agentic',
                                index_name=indexes[
                                    'email_communications']
                            )
    email_results = email_index.similarity_search(
                                                    query_text=user_question,
                                                    columns=["content", "doc_uri"],
                                                    num_results=num_results
                                                )
    contexts['email_communications'] = format_results(email_results)
    
    return contexts

def ask_llm(client, llm_endpoint_name, prompt):
    """
    Send a prompt to a Foundation Model using MLflow
    """
    # Using: databricks-meta-llama-3-3-70b-instruct
    response = client.predict(
                                endpoint=llm_endpoint_name,
                                inputs={
                                    "messages": [
                                                    {
                                                        "role": "user",
                                                        "content": prompt
                                                    }
                                                ],
                                    "temperature": 0.1,
                                    "max_tokens": 1000
                                }
                            )

    # Extract and display the response
    llm_answer = response.choices[0]['message']['content']
    print("\n" + "="*80)
    print("LLM RESPONSE:")
    print("="*80)
    print(llm_answer)
    print("="*80)


def rag_query(llm_endpoint_name, user_question, vs_endpoint_name, indexes):
    # Initialize vectoe search client
    vsc = VectorSearchClient()

    # Retrieve the context
    contexts = retrieve_contexts(vsc, vs_endpoint_name, indexes, user_question)

    # Agument the prompt for the LLM
    prompt = generate_rag_prompt(
                                    user_question=user_question,
                                    customer_feedback_context=contexts['customer_feedback'],
                                    meeting_notes_context=contexts['meeting_notes'],
                                    email_context=contexts['email_communications']
                                )

    # Initialize mlflow client
    client = get_deploy_client("databricks")

    # Send the prompt to the LLM
    ask_llm(client, llm_endpoint_name, prompt)

In [0]:
rag_query(llm_endpoint_name, user_question, vs_endpoint_name, indexes)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

LLM RESPONSE:
Based on the analysis of the context from all three sources, the main customer concerns about product delivery can be identified as follows:

1. 

## MLflow GenAI Evaluation Setup

This section integrates MLflow GenAI evaluation into the RAG system.

In [0]:
import mlflow
from mlflow.genai.scorers import RetrievalRelevance, RelevanceToQuery, Safety, Guidelines, Correctness         # Checks if answer is factually correct vs expected_facts
from mlflow.entities import Document

In [0]:
@mlflow.trace(span_type="RETRIEVER")
def retrieve_contexts_traced(vsc, vs_endpoint_name, indexes, user_question: str, num_results: int = 5):
    """
    Retrieve relevant context from all three indexes with MLflow tracing
    """
    contexts = {}
    retrieved_docs = []
    
    # Query customer feedback index
    feedback_index = vsc.get_index(
                                    endpoint_name=vs_endpoint_name,
                                    index_name=indexes['customer_feedback']
                                )
    feedback_results = feedback_index.similarity_search(
                                                        query_text=user_question,
                                                        columns=["content", "doc_uri"],
                                                        num_results=num_results
                                                    )
    contexts['customer_feedback'] = format_results(feedback_results)
    
    # Store as Documents for retrieval relevance scoring
    if feedback_results and 'result' in feedback_results and 'data_array' in feedback_results['result']:
        for row in feedback_results['result']['data_array']:
            content = row[0] if len(row) > 0 else ""
            if content:  # Only add non-empty documents
                retrieved_docs.append(Document(
                    page_content=content,
                    metadata={"source": row[1] if len(row) > 1 else "customer_feedback"}
                ))
    
    # Query meeting notes index
    meeting_index = vsc.get_index(
                                    endpoint_name=vs_endpoint_name,
                                    index_name=indexes['meeting_notes']
                                )
    meeting_results = meeting_index.similarity_search(
                                                        query_text=user_question,
                                                        columns=["content", "doc_uri"],
                                                        num_results=num_results
                                                    )
    contexts['meeting_notes'] = format_results(meeting_results)
    
    if meeting_results and 'result' in meeting_results and 'data_array' in meeting_results['result']:
        for row in meeting_results['result']['data_array']:
            content = row[0] if len(row) > 0 else ""
            if content:  # Only add non-empty documents
                retrieved_docs.append(Document(
                    page_content=content,
                    metadata={"source": row[1] if len(row) > 1 else "meeting_notes"}
                ))
    
    # Query email communications index
    email_index = vsc.get_index(
                                endpoint_name=vs_endpoint_name,
                                index_name=indexes['email_communications']
                            )
    email_results = email_index.similarity_search(
                                                    query_text=user_question,
                                                    columns=["content", "doc_uri"],
                                                    num_results=num_results
                                                )
    contexts['email_communications'] = format_results(email_results)
    
    if email_results and 'result' in email_results and 'data_array' in email_results['result']:
        for row in email_results['result']['data_array']:
            content = row[0] if len(row) > 0 else ""
            if content:  # Only add non-empty documents
                retrieved_docs.append(Document(
                    page_content=content,
                    metadata={"source": row[1] if len(row) > 1 else "email_communications"}
                ))
    
    # Store documents in span attributes for retrieval scoring
    span = mlflow.get_current_active_span()
    if span:
        span.set_attributes({"mlflow.documents": retrieved_docs})
    
    return contexts


@mlflow.trace
def rag_query_traced(user_question):
    """
    Execute RAG query with full MLflow tracing
    """
    # Initialize vector search client
    vsc = VectorSearchClient()
    
    # Retrieve contexts with tracing
    contexts = retrieve_contexts_traced(vsc, vs_endpoint_name, indexes, user_question)
    
    # Generate prompt
    prompt = generate_rag_prompt(
        user_question=user_question,
        customer_feedback_context=contexts['customer_feedback'],
        meeting_notes_context=contexts['meeting_notes'],
        email_context=contexts['email_communications']
    )
    
    # Initialize MLflow client
    client = get_deploy_client("databricks")
    
    # Query LLM
    response = client.predict(
        endpoint=llm_endpoint_name,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            "max_tokens": 1000
        }
    )
    
    llm_answer = response.choices[0]['message']['content']
    
    # Print response
    print("\n" + "="*80)
    print("LLM RESPONSE:")
    print("="*80)
    print(llm_answer)
    print("="*80)
    
    return {
        "response": llm_answer,
        "contexts": contexts,
        "prompt": prompt
    }

In [0]:
# Test the traced RAG query function
test_question = "What are the main customer concerns about product delivery?"
result = rag_query_traced(test_question)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

LLM RESPONSE:
Based on the analysis of the context from all three sources, the main customer concerns about product delivery can be identified as follows:

1. 

Trace(trace_id=tr-84aba08975786ee207ad15d5cfc39371)

In [0]:
# Load a few evaluation questions from the dataset
eval_questions_df = spark.table("rag_agentic.workday_demos.rag_evaluation_dataset").limit(10).toPandas()

# Prepare evaluation data with inputs and expectations
eval_data = [
    {
        "inputs": {"user_question": row['question']},
        "expectations": {
            "expected_facts": row['expected_answer_contains'] if isinstance(row['expected_answer_contains'], list) else [row['expected_answer_contains']]
        }
    }
    for _, row in eval_questions_df.iterrows()
]

# Define predict function for MLflow
def predict_fn(user_question: str) -> str:
    """Wrapper function for MLflow evaluation"""
    result = rag_query_traced(user_question)
    return result["response"]

# Run MLflow evaluation
print("Starting MLflow GenAI evaluation...\n")

eval_result = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        RetrievalRelevance(),  # Evaluates if retrieved docs are relevant to the query
        RelevanceToQuery(),     # Evaluates if the response answers the question
        Safety(),               # Evaluates response safety
        Correctness(),            # Checks if answer is factually correct vs expected_facts
        Guidelines(             # Custom criteria for RAG quality
            name="workday_sales_groundedness",
            guidelines=[
                "The response must be based only on the retrieved context from customer feedback, meeting notes, and email communications.",
                "The response should cite specific sources (feedback IDs, meeting IDs, or email IDs) when making claims.",
                "The response should not introduce facts not present in the knowledge base.",
                "The response should be concise and directly answer the sales question.",
            ],
        ),
    ],
)

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

Starting MLflow GenAI evaluation...



2026/09/05 14:10:07 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

LLM RESPONSE:
After analyzing the context from all three sources, I found that the common negative sentiments in customer feedback are related to the sales pro

Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T


EVALUATION COMPLETE


In [0]:
# Display evaluation metrics summary
print("\n" + "="*80)
print("📊 EVALUATION METRICS SUMMARY")
print("="*80)

metrics_df = eval_result.metrics
print("\n✅ Aggregate Metrics:")
for metric, value in metrics_df.items():
    print(f"  • {metric}: {value:.2f}" if isinstance(value, float) else f"  • {metric}: {value}")

# Display per-question results
print("\n" + "="*80)
print("📋 PER-QUESTION RESULTS")
print("="*80)

# Get the results table
results_df = eval_result.tables['eval_results']

# Display key columns
print("\n📝 Evaluation Scores by Question:\n")
for idx, row in results_df.iterrows():
    request = row['request'][0] if isinstance(row['request'], list) and len(row['request']) > 0 else "N/A"
    print(f"\nQuestion {idx+1}: {request}")
    print(f"  • Relevance to Query: {row['relevance_to_query/value']}")
    print(f"  • Workday Sales Groundedness: {row['workday_sales_groundedness/value']}")
    print(f"  • Safety: {row['safety/value']}")
    print(f"  • Execution Duration: {row['execution_duration']}ms")

print("\n" + "="*80)
print("🎯 EVALUATION COMPLETE")
print("="*80)


📊 EVALUATION METRICS SUMMARY

✅ Aggregate Metrics:
  • relevance_to_query/mean: 1.00
  • safety/mean: 1.00
  • correctness/mean: 0.40
  • workday_sales_groundedness/mean: 0.90

📋 PER-QUESTION RESULTS

📝 Evaluation Scores by Question:


Question 1: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 8594ms

Question 2: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 8671ms

Question 3: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 10740ms

Question 4: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 7456ms

Question 5: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 9587ms

Question 6: N/A
  • Relevance to Query: yes
  • Workday Sales Groundedness: yes
  • Safety: yes
  • Execution Duration: 85

## Model Registration & Deployment

This section packages the RAG pipeline as an MLflow model and deploys it to a serving endpoint.

## Model Registration & Deployment

This section packages the RAG pipeline as an MLflow model and deploys it to a serving endpoint.

### 🛠️ Code Optimizations Applied

**1. Production-Ready Model Packaging:**
- Wrapped RAG pipeline in MLflow `PythonModel` for standardized serving
- Added proper signature and input examples (required for UC registration)
- Included comprehensive error handling and graceful degradation
- Validated inputs to prevent empty/malformed queries

**2. Enhanced Error Handling:**
- Try-catch blocks around vector search operations
- Graceful fallback when retrieval fails
- Informative error messages in responses
- Defensive programming throughout

**3. Model Artifact Management:**
- Explicit pip requirements for reproducibility
- Signature inference from input/output examples
- Metadata tracking (run ID, model URI, parameters)

**4. Deployment Best Practices:**
- Workload type selection based on model requirements
- Configuration validation before deployment
- Readiness checks with timeout handling
- Comprehensive deployment status reporting

**5. Production Readiness:**
- Unity Catalog three-level namespace (catalog.schema.model)
- Model versioning and alias support
- Environment-specific configurations
- Testing and validation before registration

### 📊 Original Code vs Optimized Comparison

| Aspect | Original | Optimized |
|--------|----------|----------|
| **Deployment** | Manual functions only | MLflow PythonModel wrapper |
| **Error Handling** | None | Comprehensive try-catch |
| **Input Validation** | None | Validates empty/malformed inputs |
| **Retrieval** | Sequential | Same (optimization: add threading) |
| **Configuration** | Hardcoded | Externalized in model |
| **Monitoring** | Manual | MLflow tracking + inference tables |
| **Versioning** | None | UC registration with aliases |

### 🚀 Future Optimization Opportunities

1. **Parallel Retrieval:** Use `ThreadPoolExecutor` to query 3 indexes concurrently
2. **Caching:** Implement context caching for repeated queries
3. **Batch Processing:** Optimize for batch inference workloads
4. **Prompt Optimization:** A/B test different prompt templates
5. **Reranking:** Add cross-encoder reranker after retrieval
6. **Hybrid Search:** Combine vector + keyword search

In [0]:
# Use lazy imports to avoid circular import issues with mlflow
# Import only what's needed at class definition time
import pandas as pd
from typing import Dict, Any

# Defer mlflow imports until runtime
import sys
if 'mlflow.pyfunc' in sys.modules:
    from mlflow.pyfunc import PythonModel
else:Split 
    # If mlflow.pyfunc not available, define a base class
    class PythonModel:
        pass


class WorkdaySalesRAG(PythonModel):
    """
    Custom MLflow model that wraps the Workday Sales RAG pipeline.
    Retrieves context from vector search indexes and generates responses using LLM.
    """
    
    def load_context(self, context):
        """
        Initialize model dependencies with configuration precedence:
        1. Environment variables (highest - runtime override)
        2. Model artifacts (config.json - versioned with model)
        3. Hardcoded defaults (fallback)
        
        Handles both PAT token and Service Principal OAuth authentication.
        """
        import os
        import json
        
        # Setup authentication for model serving context
        self._setup_authentication()
        
        # Default configuration (fallback)
        default_config = {
            "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
            "vs_endpoint_name": "sales-endpoint-rag_agentic",
            "indexes": {
                'customer_feedback': 'rag_agentic.workday_demos.customer_feedback_index',
                'meeting_notes': 'rag_agentic.workday_demos.meeting_notes_index',
                'email_communications': 'rag_agentic.workday_demos.email_communications_index'
            },
            "num_results": 5,
            "temperature": 0.1,
            "max_tokens": 1000
        }
        
        # Try loading config from artifact (if provided during logging)
        config = default_config.copy()
        if context and context.artifacts:
            config_path = context.artifacts.get("config")
            if config_path:
                try:
                    with open(config_path, 'r') as f:
                        artifact_config = json.load(f)
                        config.update(artifact_config)
                        print("✅ Loaded configuration from artifact")
                except Exception as e:
                    print(f"⚠️ Could not load config artifact: {e}. Using defaults.")
        
        # Override with environment variables (highest precedence)
        self.llm_endpoint_name = os.getenv("LLM_ENDPOINT_NAME", config["llm_endpoint_name"])
        self.vs_endpoint_name = os.getenv("VS_ENDPOINT_NAME", config["vs_endpoint_name"])
        
        # Handle indexes - can be overridden individually
        self.indexes = {
            'customer_feedback': os.getenv(
                "CUSTOMER_FEEDBACK_INDEX", 
                config["indexes"]["customer_feedback"]
            ),
            'meeting_notes': os.getenv(
                "MEETING_NOTES_INDEX", 
                config["indexes"]["meeting_notes"]
            ),
            'email_communications': os.getenv(
                "EMAIL_COMMUNICATIONS_INDEX", 
                config["indexes"]["email_communications"]
            )
        }
        
        # Generation parameters
        self.num_results = int(os.getenv("NUM_RESULTS", config.get("num_results", 5)))
        self.temperature = float(os.getenv("TEMPERATURE", config.get("temperature", 0.1)))
        self.max_tokens = int(os.getenv("MAX_TOKENS", config.get("max_tokens", 1000)))
        
        print(f"📋 Configuration loaded:")
        print(f"   LLM Endpoint: {self.llm_endpoint_name}")
        print(f"   VS Endpoint: {self.vs_endpoint_name}")
        print(f"   Indexes: {len(self.indexes)} configured")
        print(f"   Auth: {self.auth_type}")
    
    def _setup_authentication(self):
        """
        Configure authentication for Databricks resources.
        Supports both PAT token and Service Principal OAuth M2M.
        """
        import os
        
        # Check for Service Principal credentials
        client_id = os.getenv("DATABRICKS_CLIENT_ID")
        client_secret = os.getenv("DATABRICKS_CLIENT_SECRET")
        host = os.getenv("DATABRICKS_HOST")
        
        if client_id and client_secret and host:
            # Service Principal OAuth M2M authentication
            self.auth_type = "Service Principal OAuth"
            self._setup_sp_auth(host, client_id, client_secret)
        elif os.getenv("DATABRICKS_TOKEN") and host:
            # PAT token authentication
            self.auth_type = "PAT Token"
            os.environ["DATABRICKS_HOST"] = host
        else:
            # Default/fallback authentication (notebook context)
            self.auth_type = "Default (Notebook)"
            print("⚠️ Running in notebook context, using default authentication")
    
    def _setup_sp_auth(self, host, client_id, client_secret):
        """
        Setup Service Principal OAuth M2M authentication.
        Obtains OAuth token and configures environment.
        """
        import requests
        import os
        
        try:
            # Get OAuth access token using client credentials flow
            token_url = f"{host}/oidc/v1/token"
            
            response = requests.post(
                token_url,
                data={
                    "grant_type": "client_credentials",
                    "scope": "all-apis"
                },
                auth=(client_id, client_secret),
                headers={"Content-Type": "application/x-www-form-urlencoded"},
                timeout=30
            )
            
            if response.status_code == 200:
                token_data = response.json()
                access_token = token_data.get("access_token")
                
                if access_token:
                    # Set environment variables for Databricks SDK
                    os.environ["DATABRICKS_HOST"] = host
                    os.environ["DATABRICKS_TOKEN"] = access_token
                    print("✅ Service Principal OAuth authentication configured")
                else:
                    raise ValueError("No access token in OAuth response")
            else:
                raise ValueError(f"OAuth token request failed: {response.status_code} - {response.text}")
                
        except Exception as e:
            print(f"❌ Service Principal authentication failed: {str(e)}")
            print("   Falling back to default authentication")
            raise
        
    def _format_results(self, results):
        """Format search results into readable context"""
        if not results or 'result' not in results or 'data_array' not in results['result']:
            return "No relevant information found."
        
        formatted = []
        for i, row in enumerate(results['result']['data_array'], 1):
            content = row[0] if len(row) > 0 else "N/A"
            doc_uri = row[1] if len(row) > 1 else "Unknown source"
            formatted.append(f"[{i}] Source: {doc_uri}\n{content}\n")
        
        return "\n".join(formatted)
    
    def _retrieve_contexts(self, user_question: str, num_results: int = None) -> Dict[str, str]:
        """Retrieve context from all three vector indexes"""
        from databricks.ai_search.client import VectorSearchClient
        
        # Use configured num_results if not provided
        if num_results is None:
            num_results = self.num_results
        
        vsc = VectorSearchClient()
        contexts = {}
        
        try:
            # Customer feedback
            feedback_index = vsc.get_index(
                endpoint_name=self.vs_endpoint_name,
                index_name=self.indexes['customer_feedback']
            )
            feedback_results = feedback_index.similarity_search(
                query_text=user_question,
                columns=["content", "doc_uri"],
                num_results=num_results
            )
            contexts['customer_feedback'] = self._format_results(feedback_results)
            
            # Meeting notes
            meeting_index = vsc.get_index(
                endpoint_name=self.vs_endpoint_name,
                index_name=self.indexes['meeting_notes']
            )
            meeting_results = meeting_index.similarity_search(
                query_text=user_question,
                columns=["content", "doc_uri"],
                num_results=num_results
            )
            contexts['meeting_notes'] = self._format_results(meeting_results)
            
            # Email communications
            email_index = vsc.get_index(
                endpoint_name=self.vs_endpoint_name,
                index_name=self.indexes['email_communications']
            )
            email_results = email_index.similarity_search(
                query_text=user_question,
                columns=["content", "doc_uri"],
                num_results=num_results
            )
            contexts['email_communications'] = self._format_results(email_results)
            
        except Exception as e:
            # Graceful degradation
            contexts = {
                'customer_feedback': f"Error retrieving customer feedback: {str(e)}",
                'meeting_notes': f"Error retrieving meeting notes: {str(e)}",
                'email_communications': f"Error retrieving emails: {str(e)}"
            }
        
        return contexts
    
    def _generate_prompt(self, user_question: str, contexts: Dict[str, str]) -> str:
        """Generate RAG prompt with retrieved contexts"""
        prompt = f"""You are an intelligent assistant with access to information from customer feedback, meeting notes, and email communications.

## Retrieved Context:

### Customer Feedback:
{contexts['customer_feedback']}

### Meeting Notes:
{contexts['meeting_notes']}

### Email Communications:
{contexts['email_communications']}

## User Question:
{user_question}

## Instructions:
1. Analyze the context from all three sources carefully
2. Synthesize information across customer feedback, meeting notes, and emails to provide a comprehensive answer
3. If information conflicts between sources, note the discrepancy and provide context
4. If the context doesn't contain relevant information, acknowledge this limitation
5. Cite which source(s) your answer comes from (customer feedback, meeting notes, or emails)
6. Provide specific examples or quotes when relevant

## Answer:
"""
        return prompt
    
    def predict(self, context, model_input):
        """
        Generate RAG responses for input questions.
        
        Args:
            model_input: pandas DataFrame with 'question' column
            
        Returns:
            pandas DataFrame with 'question' and 'answer' columns
        """
        from mlflow.deployments import get_deploy_client
        
        # Handle both DataFrame and dict inputs
        if isinstance(model_input, pd.DataFrame):
            questions = model_input['question'].tolist()
        elif isinstance(model_input, dict):
            questions = [model_input.get('question', '')]
        else:
            questions = [str(model_input)]
        
        # Validate inputs
        if not questions or all(not q.strip() for q in questions):
            return pd.DataFrame({
                'question': questions,
                'answer': ['Error: Empty question provided'] * len(questions)
            })
        
        client = get_deploy_client("databricks")
        answers = []
        
        for question in questions:
            try:
                # Retrieve contexts
                contexts = self._retrieve_contexts(question)
                
                # Generate prompt
                prompt = self._generate_prompt(question, contexts)
                
                # Query LLM with configured parameters
                response = client.predict(
                    endpoint=self.llm_endpoint_name,
                    inputs={
                        "messages": [{"role": "user", "content": prompt}],
                        "temperature": self.temperature,
                        "max_tokens": self.max_tokens
                    }
                )
                
                answer = response.choices[0]['message']['content']
                answers.append(answer)
                
            except Exception as e:
                answers.append(f"Error generating response: {str(e)}")
        
        return pd.DataFrame({
            'question': questions,
            'answer': answers
        })


print("✅ WorkdaySalesRAG model class defined")

✅ WorkdaySalesRAG model class defined


In [0]:
# Create configuration file for model artifact
import json

model_config = {
    "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
    "vs_endpoint_name": "sales-endpoint-rag_agentic",
    "indexes": {
        "customer_feedback": "rag_agentic.workday_demos.customer_feedback_index",
        "meeting_notes": "rag_agentic.workday_demos.meeting_notes_index",
        "email_communications": "rag_agentic.workday_demos.email_communications_index"
    },
    "num_results": 5,
    "temperature": 0.1,
    "max_tokens": 1000
}

# Save config to file
config_path = "/tmp/model_config.json"
with open(config_path, "w") as f:
    json.dump(model_config, f, indent=2)

print("📝 Model Configuration:")
print(json.dumps(model_config, indent=2))

# Create input example and signature
input_example = pd.DataFrame({
    'question': ['What are the main customer concerns about product delivery?']
})

# Generate sample output for signature
model_instance = WorkdaySalesRAG()
model_instance.load_context(None)

# Create a simpler output for signature (the actual predict returns a full answer)
output_example = pd.DataFrame({
    'question': ['What are the main customer concerns about product delivery?'],
    'answer': ['Based on customer feedback, meeting notes, and email communications...']
})

# Infer signature
signature = infer_signature(input_example, output_example)

print("📝 Model Signature:")
print(signature)
print("\n📦 Input Example:")
print(input_example)

# Log the model to MLflow with config artifact
with mlflow.start_run(run_name="workday_sales_rag_v1") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="workday_sales_rag_model",
        python_model=WorkdaySalesRAG(),
        artifacts={"config": config_path},  # 👈 Include config artifact
        signature=signature,
        input_example=input_example,
        pip_requirements=[
            "databricks-ai-search",
            "mlflow[databricks]",
            "pandas",
            "requests",  # For OAuth token retrieval
        ],
    )
    
    # Log parameters for tracking
    mlflow.log_params({
        "llm_endpoint": model_config["llm_endpoint_name"],
        "vs_endpoint": model_config["vs_endpoint_name"],
        "num_indexes": len(model_config["indexes"]),
        "num_results": model_config["num_results"],
        "temperature": model_config["temperature"],
        "max_tokens": model_config["max_tokens"]
    })
    
    run_id = run.info.run_id
    model_uri = model_info.model_uri

print(f"\n✅ Model logged successfully!")
print(f"   Run ID: {run_id}")
print(f"   Model URI: {model_uri}")
print(f"\n🔗 View in MLflow: {mlflow.get_tracking_uri()}/mlflow/experiments/")

📝 Model Configuration:
{
  "llm_endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
  "vs_endpoint_name": "sales-endpoint-rag_agentic",
  "indexes": {
    "customer_feedback": "rag_agentic.workday_demos.customer_feedback_index",
    "meeting_notes": "rag_agentic.workday_demos.meeting_notes_index",
    "email_communications": "rag_agentic.workday_demos.email_communications_index"
  },
  "num_results": 5,
  "temperature": 0.1,
  "max_tokens": 1000
}


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6714315719604261>, line 26
     23 print(json.dumps(model_config, indent=2))
     25 # Create input example and signature
---> 26 input_example = pd.DataFrame({
     27     'question': ['What are the main customer concerns about product delivery?']
     28 })
     30 # Generate sample output for signature
     31 model_instance = WorkdaySalesRAG()

NameError: name 'pd' is not defined

In [0]:
import mlflow
import pandas as pd

# Get the model_uri from the previous run
model_uri = "models:/m-2075cbcefb374870a5f8908fe0a36480"

# Validate the logged model before registration
print("\n🔍 Validating model before registration...\n")
print(f"   Model URI: {model_uri}\n")

# Load the model
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

# Check signature
if pyfunc_model.metadata.signature is None:
    raise ValueError("❌ The logged model has no signature; fix the original artifact before registration.")
print("✅ Signature validation passed")

# Check input example
representative_input = pyfunc_model.input_example
if representative_input is None:
    raise ValueError("❌ The logged model has no input example; fix the original artifact before registration.")
print("✅ Input example validation passed")

# Test prediction with input example
print("\n🧪 Testing model prediction with sample input...")
try:
    test_result = mlflow.models.predict(
        model_uri=model_uri,
        input_data=representative_input,
    )
    print("✅ Model prediction test passed")
    print("\n📊 Sample Prediction Result:")
    print(test_result)
except Exception as e:
    raise ValueError(f"❌ Model prediction failed: {str(e)}")

print("\n✨ Model validation complete! Ready for registration.")


🔍 Validating model before registration...

   Model URI: models:/m-2075cbcefb374870a5f8908fe0a36480



2026/09/05 23:22:29 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.


✅ Loaded configuration from artifact
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
✅ Signature validation passed
✅ Input example validation passed

🧪 Testing model prediction with sample input...


2026/09/05 23:22:31 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/09/05 23:22:33 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
-> https://www.python.org/ftp/python/3.12.3/Python-3.12.3.tar.xz
Installing Python-3.12.3...
Installed Python-3.12.3 to /tmp/pyenv_root/versions/3.12.3
2026/09/05 23:25:16 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-ef286782e0b6d52d339a96f7085b2ce9427117a3 with /tmp/pyenv_root/versions/3.12.3/bin/python
2026/09/05 23:25:20 INFO mlflow.utils.virtualenv: Installing dependencies


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 142.3 MB/s eta 0:00:00
   


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
2026/09/05 23:26:18 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-ef286782e0b6d52d339a96f7085b2ce9427117a3/bin/activate && python -c ""']'
2026/09/05 23:26:18 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-ef286782e0b6d52d339a96f7085b2ce9427117a3/bin/activate && python /local_disk0/.ephemeral_nfs/envs/pythonEnv-d1918804-8850-444c-9376-79a1640a853c/lib/python3.12/site-packages/mlflow/pyfunc/_mlflow_pyfunc_backend_predict.py --model-uri file:///local_disk0/user_tmp_data/spark-d1918804-8850-444c-9376-79/tmpauxuvqih --content-type json --input-path /local_disk0/user_tmp_data/spark-d1918804-8850-444c-9376-79/tmp66u6qzfi/input.json']'
2026/09/05 23:26:21 WARNING mlflow.pyfunc: The version of CloudPickle that was used to save the model, `CloudPickle 3.0.0`, differs from 

✅ Loaded configuration from artifact
📋 Configuration loaded:
   LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   VS Endpoint: sales-endpoint-rag_agentic
   Indexes: 3 configured
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, p

In [0]:
# Unity Catalog registration configuration
# 👉 UPDATE THESE VALUES for your environment:
catalog_name = "rag_agentic"  # Your Unity Catalog name
schema_name = "workday_demos"  # Your schema name
model_name = "workday_sales_rag"  # Model name

# Full three-level namespace
registered_model_name = f"{catalog_name}.{schema_name}.{model_name}"

print(f"\n📦 Registering model to Unity Catalog...")
print(f"   Target: {registered_model_name}")
print(f"   Source: {model_uri}")

# Register the model
mlflow.set_registry_uri("databricks-uc")
registered_version = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name,
    await_registration_for=300,  # Wait up to 5 minutes
)

model_version = registered_version.version

print(f"\n✅ Model registered successfully!")
print(f"   Model: {registered_model_name}")
print(f"   Version: {model_version}")
print(f"   URI: models:/{registered_model_name}/{model_version}")

# Optional: Set alias (e.g., 'champion' for production)
set_alias = True  # Set to False to skip
alias_name = "champion"

if set_alias:
    from mlflow import MlflowClient
    
    registry_client = MlflowClient(registry_uri="databricks-uc")
    registry_client.set_registered_model_alias(
        name=registered_model_name,
        alias=alias_name,
        version=model_version,
    )
    print(f"\n🏆 Alias '{alias_name}' set to version {model_version}")
    print(f"   Can now reference as: models:/{registered_model_name}@{alias_name}")


📦 Registering model to Unity Catalog...
   Target: rag_agentic.workday_demos.workday_sales_rag
   Source: models:/m-2075cbcefb374870a5f8908fe0a36480


Successfully registered model 'rag_agentic.workday_demos.workday_sales_rag'.


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '1' of model 'rag_agentic.workday_demos.workday_sales_rag': https://dbc-00781aa6-52b5.cloud.databricks.com/explore/data/models/rag_agentic/workday_demos/workday_sales_rag/version/1?o=7474652423374021



✅ Model registered successfully!
   Model: rag_agentic.workday_demos.workday_sales_rag
   Version: 1
   URI: models:/rag_agentic.workday_demos.workday_sales_rag/1

🏆 Alias 'champion' set to version 1
   Can now reference as: models:/rag_agentic.workday_demos.workday_sales_rag@champion


In [0]:
from databricks.sdk import WorkspaceClient

# Initialize workspace client
w = WorkspaceClient()

# Fetch supported workload types and sizes for this workspace
print("📊 Fetching supported workload types...\n")
workload_response = w.api_client.do(
    "GET", "/api/2.0/serving-endpoints:workload-configs"
)
supported_workloads = {
    config["workload_type"]: [size["key"] for size in config.get("workload_sizes", [])]
    for config in workload_response.get("workload_configs", [])
}

print("✅ Supported Workload Configurations:")
for workload_type, sizes in supported_workloads.items():
    print(f"   {workload_type}: {', '.join(sizes)}")

# Model requirements analysis
print("\n🔍 Model Requirements Analysis:")
print("   • Model Type: RAG pipeline (LLM + Vector Search)")
print("   • Compute Needed: CPU (no CUDA tensors, no GPU operations)")
print("   • Memory: Low (model is stateless, delegates to endpoints)")
print("   • Dependencies: databricks-ai-search, mlflow")

# Recommended configuration
workload_type = "CPU"  # No GPU required for this RAG pipeline
workload_size = "Small"  # Start small, scale up if needed
scale_to_zero_enabled = True  # Enable to save costs during idle periods

print("\n🎯 Recommended Deployment Configuration:")
print(f"   • Workload Type: {workload_type}")
print(f"   • Workload Size: {workload_size}")
print(f"   • Scale to Zero: {scale_to_zero_enabled}")
print("\n⚠️  Note: First request after inactivity will have cold start latency (~1-2 min)")

📊 Fetching supported workload types...

✅ Supported Workload Configurations:
   CPU: Small, Medium, Large
   CPU_MEDIUM: Small, Medium, Large
   CPU_LARGE: Small, Medium, Large
   GPU_MEDIUM: Small, Medium, Large
   MULTIGPU_MEDIUM: Small, Medium, Large
   GPU_SMALL: Small, Medium, Large
   GPU_MEDIUM_8: Small, Medium, Large
   GPU_LARGE: Small, Medium, Large

🔍 Model Requirements Analysis:
   • Model Type: RAG pipeline (LLM + Vector Search)
   • Compute Needed: CPU (no CUDA tensors, no GPU operations)
   • Memory: Low (model is stateless, delegates to endpoints)
   • Dependencies: databricks-ai-search, mlflow

🎯 Recommended Deployment Configuration:
   • Workload Type: CPU
   • Workload Size: Small
   • Scale to Zero: True

⚠️  Note: First request after inactivity will have cold start latency (~1-2 min)


In [0]:
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    ServingModelWorkloadType,
)

# Endpoint configuration
endpoint_name = f"{model_name}_endpoint"  # workday_sales_rag_endpoint

print(f"\n🚀 Endpoint Deployment Plan")
print("=" * 60)
print(f"   Endpoint Name: {endpoint_name}")
print(f"   Model: {registered_model_name}")
print(f"   Version: {model_version}")
print(f"   Workload: {workload_type} / {workload_size}")
print(f"   Scale to Zero: {scale_to_zero_enabled}")
print("\n⏱️  Deployment typically takes ~15 minutes")
print("=" * 60)

# 👉 IMPORTANT: User must confirm before proceeding
confirm_deployment = True  # Set to True after reviewing configuration

if not confirm_deployment:
    print("\n⚠️  Deployment paused. Set confirm_deployment=True to proceed.")
    raise Exception("User confirmation required before deployment")

print("\n✅ Configuration confirmed. Starting deployment...\n")

# Build served entity configuration with authentication
# 🔑 IMPORTANT: Configure authentication for Databricks resources
print("\n🔐 Authentication Configuration:")
print("   The model needs credentials to access LLM and Vector Search endpoints")
print("   Using workspace authentication (suitable for development)")
print("   For production, use service principal tokens stored in secrets\n")

# Environment variables for Service Principal authentication
# Using OAuth M2M (Machine-to-Machine) authentication with service principal
# Secrets stored in 'agentic' scope: sp_client_id, sp_client_secret

auth_env_vars = {
    "DATABRICKS_HOST": f"https://{w.config.host}",
    "DATABRICKS_CLIENT_ID": "{{secrets/agentic/sp_client_id}}",
    "DATABRICKS_CLIENT_SECRET": "{{secrets/agentic/sp_client_secret}}",
}

served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=str(model_version),
    workload_type=ServingModelWorkloadType(workload_type),
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero_enabled,
    environment_vars=auth_env_vars,  # 👈 Authentication config
)

endpoint_config = EndpointCoreConfigInput(served_entities=[served_entity])

# Check if endpoint already exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"🔄 Endpoint '{endpoint_name}' exists. Checking configuration...")
    
    # Validate we're not replacing a different model
    current_entities = list(
        existing_endpoint.config.served_entities
        if existing_endpoint.config and existing_endpoint.config.served_entities
        else []
    )
    current_entity_names = [entity.entity_name for entity in current_entities]
    
    if len(current_entities) != 1 or current_entity_names[0] != registered_model_name:
        print(f"\n⚠️  WARNING: Endpoint currently serves {current_entity_names}")
        print(f"   Refusing to replace with {registered_model_name}")
        print(f"\n👉 Choose a different endpoint name or confirm full replacement.")
        raise ValueError(f"Endpoint serves different model(s): {current_entity_names}")
    
    # Update existing endpoint
    print(f"✅ Updating endpoint to version {model_version}...")
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[served_entity],
    )
    print(f"\n✅ Endpoint update initiated!")
    
except NotFound:
    # Create new endpoint
    print(f"✨ Creating new endpoint '{endpoint_name}'...")
    w.serving_endpoints.create(
        name=endpoint_name,
        config=endpoint_config,
    )
    print(f"\n✅ Endpoint creation initiated!")

print(f"\n🔗 Endpoint URL will be available at:")
print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
print(f"\n⏳ Proceeding to readiness check...")


🚀 Endpoint Deployment Plan
   Endpoint Name: workday_sales_rag_endpoint
   Model: rag_agentic.workday_demos.workday_sales_rag
   Version: 1
   Workload: CPU / Small
   Scale to Zero: True

⏱️  Deployment typically takes ~15 minutes

✅ Configuration confirmed. Starting deployment...

✨ Creating new endpoint 'workday_sales_rag_endpoint'...

✅ Endpoint creation initiated!

🔗 Endpoint URL will be available at:
   https://https://dbc-00781aa6-52b5.cloud.databricks.com/ml/endpoints/workday_sales_rag_endpoint

⏳ Proceeding to readiness check...


## 🌍 Environment-Specific Deployment Configurations

### Configuration Hierarchy

The model supports **three-tier configuration** with this precedence:

1. **🔴 Environment Variables** (Highest) - Runtime override, set during endpoint creation
2. **🟡 Model Artifacts** (Medium) - `config.json` versioned with the model
3. **🟢 Hardcoded Defaults** (Lowest) - Fallback if nothing else specified

### Example: Deploy to Different Environments

#### **Development Environment**
```python
# Override with dev-specific endpoints
dev_env_vars = {
    "LLM_ENDPOINT_NAME": "databricks-meta-llama-3-3-70b-instruct",
    "VS_ENDPOINT_NAME": "sales-endpoint-rag_agentic-dev",
    "CUSTOMER_FEEDBACK_INDEX": "rag_agentic_dev.workday_demos.customer_feedback_index",
    "MEETING_NOTES_INDEX": "rag_agentic_dev.workday_demos.meeting_notes_index",
    "EMAIL_COMMUNICATIONS_INDEX": "rag_agentic_dev.workday_demos.email_communications_index",
    "NUM_RESULTS": "3",  # Faster for dev testing
    "TEMPERATURE": "0.2"
}

w.serving_endpoints.create(
    name="workday_sales_rag_dev",
    config=EndpointCoreConfigInput(
        served_entities=[served_entity],
        environment_vars=dev_env_vars  # 👈 Dev config
    )
)
```

#### **Staging Environment**
```python
# Staging with moderate settings
staging_env_vars = {
    "LLM_ENDPOINT_NAME": "databricks-meta-llama-3-3-70b-instruct",
    "VS_ENDPOINT_NAME": "sales-endpoint-rag_agentic-staging",
    "CUSTOMER_FEEDBACK_INDEX": "rag_agentic_staging.workday_demos.customer_feedback_index",
    "NUM_RESULTS": "5"
}

w.serving_endpoints.create(
    name="workday_sales_rag_staging",
    config=EndpointCoreConfigInput(
        served_entities=[served_entity],
        environment_vars=staging_env_vars  # 👈 Staging config
    )
)
```

#### **Production Environment**
```python
# Production with prod endpoints and optimized settings
prod_env_vars = {
    "LLM_ENDPOINT_NAME": "databricks-meta-llama-3-3-70b-instruct",
    "VS_ENDPOINT_NAME": "sales-endpoint-rag_agentic-prod",
    "CUSTOMER_FEEDBACK_INDEX": "rag_agentic_prod.workday_demos.customer_feedback_index",
    "MEETING_NOTES_INDEX": "rag_agentic_prod.workday_demos.meeting_notes_index",
    "EMAIL_COMMUNICATIONS_INDEX": "rag_agentic_prod.workday_demos.email_communications_index",
    "NUM_RESULTS": "5",
    "TEMPERATURE": "0.1",  # Conservative for production
    "MAX_TOKENS": "1000"
}

w.serving_endpoints.create(
    name="workday_sales_rag_prod",
    config=EndpointCoreConfigInput(
        served_entities=[served_entity],
        environment_vars=prod_env_vars,  # 👈 Prod config
        traffic_config=TrafficConfig(
            routes=[
                Route(
                    served_model_name="workday_sales_rag-prod",
                    traffic_percentage=100
                )
            ]
        )
    )
)
```

### Benefits of This Approach

✅ **Same Model Version** deployed to all environments  
✅ **Different Configurations** per environment without relogging  
✅ **Easy A/B Testing** - just update env vars  
✅ **Configuration Changes** without new model versions  
✅ **Environment Isolation** - dev/staging/prod use separate indexes  
✅ **Version Control** - config.json tracked with model  

### Updating Configuration After Deployment

```python
# Update endpoint configuration without redeploying model
w.serving_endpoints.update_config(
    name="workday_sales_rag_prod",
    served_entities=[served_entity],
    environment_vars=updated_prod_env_vars  # 👈 New config
)
```

In [0]:
import time
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady


def wait_for_endpoint_ready(name, timeout_s=900, poll_s=15):
    """
    Wait for endpoint to be ready.
    
    Args:
        name: Endpoint name
        timeout_s: Maximum wait time in seconds (default: 15 minutes)
        poll_s: Polling interval in seconds
    """
    deadline = time.time() + timeout_s
    failure_states = {
        EndpointStateConfigUpdate.UPDATE_FAILED,
        EndpointStateConfigUpdate.UPDATE_CANCELED,
    }
    
    print(f"🔄 Waiting for endpoint '{name}' to be ready...")
    print(f"   Timeout: {timeout_s}s ({timeout_s // 60} minutes)")
    print(f"   Polling every {poll_s}s\n")
    
    elapsed = 0
    while time.time() < deadline:
        state = w.serving_endpoints.get(name).state
        
        if (
            state.ready == EndpointStateReady.READY
            and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"\n✅ Endpoint '{name}' is READY!")
            print(f"   Total deployment time: {elapsed}s ({elapsed // 60} min {elapsed % 60} sec)")
            return
        
        if state.config_update in failure_states:
            print(f"\n❌ Deployment FAILED")
            print(f"   State: {state.config_update.value}")
            raise RuntimeError(
                f"{name} deployment failed with config update state {state.config_update.value}"
            )
        
        # Progress update every minute
        if elapsed % 60 == 0:
            print(f"   ⏳ Still deploying... ({elapsed}s elapsed, state: {state.config_update.value})")
        
        time.sleep(poll_s)
        elapsed += poll_s
    
    raise TimeoutError(f"❌ Endpoint '{name}' not ready after {timeout_s}s")


# Wait for readiness
try:
    wait_for_endpoint_ready(endpoint_name)
    
    # Display endpoint details
    endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"\n🎉 DEPLOYMENT SUCCESSFUL!")
    print("=" * 60)
    print(f"   Endpoint Name: {endpoint_name}")
    print(f"   State: {endpoint.state.ready.value}")
    print(f"   Model: {registered_model_name} (v{model_version})")
    print(f"\n🔗 Endpoint URL:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    print("\n📡 Ready to serve predictions!")
    print("=" * 60)
    
except (TimeoutError, RuntimeError) as e:
    print(f"\n❌ Deployment Error: {str(e)}")
    print(f"\n👉 Check endpoint status at:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    raise

🔄 Waiting for endpoint 'workday_sales_rag_endpoint' to be ready...
   Timeout: 900s (15 minutes)
   Polling every 15s


✅ Endpoint 'workday_sales_rag_endpoint' is READY!
   Total deployment time: 0s (0 min 0 sec)

🎉 DEPLOYMENT SUCCESSFUL!
   Endpoint Name: workday_sales_rag_endpoint
   State: READY
   Model: rag_agentic.workday_demos.workday_sales_rag (v1)

🔗 Endpoint URL:
   https://https://dbc-00781aa6-52b5.cloud.databricks.com/ml/endpoints/workday_sales_rag_endpoint

📡 Ready to serve predictions!


In [0]:
# Test the deployed endpoint
print("🧪 Testing deployed endpoint...\n")

# Prepare test input - match the model signature format
import pandas as pd

test_questions = pd.DataFrame({
    'question': [
        "What are the main customer concerns about product delivery?",
        "What positive feedback has been received about the product demo?"
    ]
})

try:
    # Query the endpoint with DataFrame input
    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_records=test_questions.to_dict(orient='records')
    )
    
    print("✅ Endpoint Test Successful!\n")
    print("="*80)
    print("TEST RESULTS")
    print("="*80)
    
    # Display predictions
    predictions = response.predictions
    for i, pred in enumerate(predictions, 1):
        print(f"\n💬 Question {i}:")
        print(pred.get('question', 'N/A'))
        print(f"\n📝 Answer {i}:")
        print(pred.get('answer', 'N/A'))
        print("\n" + "-"*80)
    
    print("\n✨ Endpoint is serving predictions correctly!")
    
except Exception as e:
    print(f"❌ Endpoint test failed: {str(e)}")
    print("\n👉 Troubleshooting steps:")
    print("   1. Verify endpoint is in READY state")
    print("   2. Check endpoint logs for errors")
    print("   3. Validate model dependencies")
    print(f"   4. Check endpoint at: https://{w.config.host}/ml/endpoints/{endpoint_name}")
    raise

🧪 Testing deployed endpoint...

✅ Endpoint Test Successful!

TEST RESULTS

💬 Question 1:
What are the main customer concerns about product delivery?

📝 Answer 1:
Error generating response: Reading Databricks credential configuration in model serving failed. Most commonly, this happens because the model currently being served was logged without Databricks resource dependencies properly specified. Re-log your model, specifying resource dependencies as described in https://docs.databricks.com/en/generative-ai/agent-framework/log-agent.html#specify-resources-for-pyfunc-or-langchain-agent and then register and attempt to serve it again. Alternatively, you can explicitly configure authentication by setting environment variables as described in https://docs.databricks.com/en/generative-ai/agent-framework/deploy-agent.html#manual-authentication. Additional debug info: the MLflow tracking URI was set to 'None'

--------------------------------------------------------------------------------

💬 

In [0]:
# =============================================================================
# FIX AUTHENTICATION ISSUE
# =============================================================================
# The model needs credentials to access Databricks endpoints (LLM + Vector Search)
# when running in Model Serving. We'll set up authentication step-by-step.
# =============================================================================

from databricks.sdk.service.serving import ServedEntityInput
import warnings

print("🔧 AUTHENTICATION FIX FOR MODEL SERVING")
print("="*80)
print("\n🐞 Issue: The model cannot authenticate to access:")
print("   • LLM Endpoint: databricks-meta-llama-3-3-70b-instruct")
print("   • Vector Search Endpoint: sales-endpoint-rag_agentic")
print("   • Vector Search Indexes (customer_feedback, meeting_notes, email_communications)")
print("\n💡 Solution: Add authentication via environment variables")
print("="*80)

# Step 1: Generate a Personal Access Token
print("\n🔑 STEP 1: Generate Personal Access Token")
print("-"*80)
print("1. Click your profile icon (top right) > User Settings")
print("2. Go to 'Developer' tab")
print("3. Click 'Manage' under 'Access tokens'")
print("4. Click 'Generate new token'")
print("5. Set comment: 'RAG Model Serving' and lifetime: 90 days")
print("6. Copy the token (you'll need it in Step 2)")
print("")
print("⚠️  SAVE THE TOKEN - You won't be able to see it again!")

# Step 2: Store token in Databricks Secrets
print("\n🔐 STEP 2: Create Secret Scope and Store Token")
print("-"*80)
print("Run these commands in a NEW notebook cell or terminal:\n")
print("# Create a secret scope (one-time setup)")
print("dbutils.secrets.help()")
print("")
print("# Since we can't use dbutils.secrets.put in notebooks, use the Databricks CLI:")
print("# 1. Install CLI: pip install databricks-cli")
print("# 2. Configure: databricks configure --token")
print("# 3. Create scope: databricks secrets create-scope --scope rag_model_serving")
print("# 4. Put secret: databricks secrets put --scope rag_model_serving --key databricks_token")
print("")
print("OR use the Databricks UI:")
print(f"1. Go to: https://{w.config.host}#secrets/createScope")
print("2. Scope name: rag_model_serving")
print("3. Then use CLI to add token: databricks secrets put --scope rag_model_serving --key databricks_token")

# Step 3: Update endpoint with authentication
print("\n🚀 STEP 3: Update Endpoint Configuration")
print("-"*80)
print("After setting up the secret, run this to update the endpoint:\n")

# Prepare the update configuration
auth_env_vars = {
    "DATABRICKS_HOST": f"https://{w.config.host}",
    "DATABRICKS_TOKEN": "{{secrets/rag_model_serving/databricks_token}}"
}

print("# Configuration to add:")
print(f"auth_env_vars = {auth_env_vars}")
print("")
print("# Update the endpoint:")
print("from databricks.sdk.service.serving import ServedEntityInput")
print("")
print("served_entity_with_auth = ServedEntityInput(")
print(f"    entity_name='{registered_model_name}',")
print(f"    entity_version='{model_version}',")
print("    workload_type='CPU',")
print(f"    workload_size='{workload_size}',")
print(f"    scale_to_zero_enabled={scale_to_zero_enabled},")
print("    environment_vars=auth_env_vars  # 👈 Authentication here")
print(")")
print("")
print(f"w.serving_endpoints.update_config(")
print(f"    name='{endpoint_name}',")
print("    served_entities=[served_entity_with_auth]")
print(")")

print("\n" + "="*80)
print("✅ QUICK START - Copy this complete cell below:")
print("="*80)
print(f"""
# After setting up secrets, run this cell to update endpoint with auth:

from databricks.sdk.service.serving import ServedEntityInput

auth_env_vars = {{
    "DATABRICKS_HOST": "https://{w.config.host}",
    "DATABRICKS_TOKEN": "{{{{secrets/rag_model_serving/databricks_token}}}}"
}}

served_entity_with_auth = ServedEntityInput(
    entity_name="{registered_model_name}",
    entity_version="{model_version}",
    workload_type="CPU",
    workload_size="{workload_size}",
    scale_to_zero_enabled={scale_to_zero_enabled},
    environment_vars=auth_env_vars
)

print("Updating endpoint with authentication...")
w.serving_endpoints.update_config(
    name="{endpoint_name}",
    served_entities=[served_entity_with_auth]
)

print("✅ Endpoint updated! Wait ~2-3 minutes for config update to complete.")
print(f"Check status: https://{w.config.host}/ml/endpoints/{endpoint_name}")
""")

print("="*80)
print("📝 Summary:")
print("1. Generate Personal Access Token (User Settings > Developer)")
print("2. Store in secrets: databricks secrets put --scope rag_model_serving --key databricks_token")
print("3. Run the cell above to update endpoint with authentication")
print("4. Wait 2-3 minutes, then re-test the endpoint (Cell 22)")
print("="*80)

🔧 AUTHENTICATION FIX FOR MODEL SERVING

🐞 Issue: The model cannot authenticate to access:
   • LLM Endpoint: databricks-meta-llama-3-3-70b-instruct
   • Vector Search Endpoint: sales-endpoint-rag_agentic
   • Vector Search Indexes (customer_feedback, meeting_notes, email_communications)

💡 Solution: Add authentication via environment variables

🔑 STEP 1: Generate Personal Access Token
--------------------------------------------------------------------------------
1. Click your profile icon (top right) > User Settings
2. Go to 'Developer' tab
3. Click 'Manage' under 'Access tokens'
4. Click 'Generate new token'
5. Set comment: 'RAG Model Serving' and lifetime: 90 days
6. Copy the token (you'll need it in Step 2)

⚠️  SAVE THE TOKEN - You won't be able to see it again!

🔐 STEP 2: Create Secret Scope and Store Token
--------------------------------------------------------------------------------
Run these commands in a NEW notebook cell or terminal:

# Create a secret scope (one-time set

## 🔐 Deploy with Service Principal Authentication - Complete Solution

### ✅ What We Fixed

**Authentication Issue:** The model needs credentials to access:
* LLM Endpoint: `databricks-meta-llama-3-3-70b-instruct`
* Vector Search Endpoint: `sales-endpoint-rag_agentic`
* Vector Search Indexes (3 indexes)

**Solution Implemented:**
1. ✅ **Service Principal Created** - `agentic_sp` with appropriate permissions
2. ✅ **Secrets Stored** - `sp_client_id` and `sp_client_secret` in `agentic` scope
3. ✅ **Model Updated** - Added OAuth M2M authentication to `WorkdaySalesRAG` class
4. ✅ **Endpoint Config Updated** - Environment variables reference service principal secrets

---

### 📋 Deployment Checklist

Before proceeding, verify these prerequisites:

#### 1. Service Principal Permissions

Your service principal **agentic_sp** needs these permissions:

| Resource | Permission | How to Grant |
|----------|------------|-------------|
| **LLM Endpoint**<br>`databricks-meta-llama-3-3-70b-instruct` | `CAN_QUERY` | Serving → Foundation Model APIs → Click endpoint → Permissions tab → Add service principal |
| **Vector Search Endpoint**<br>`sales-endpoint-rag_agentic` | `CAN_QUERY` | Compute → Vector Search → Click endpoint → Permissions tab → Add service principal |
| **Vector Search Indexes**<br>(3 indexes in `rag_agentic.workday_demos`) | `SELECT` on source tables | Catalog → rag_agentic → workday_demos → Grant SELECT on schema or individual tables |

#### 2. Secrets Verification

Run in the [databricks_secrets](#notebook-3842118283998758) notebook:
```python
dbutils.secrets.list('agentic')
# Should show: sp_client_id, sp_client_secret
```

---

### 🚀 Deployment Steps

#### Step 1: Re-run Updated Model Definition

**Run [Cell 14](#cell-1d8c1bba-2fcb-499b-b0b9-3d553c0fdadf)** to reload the model class with OAuth authentication.

```python
# Cell 14 now includes:
# - _setup_authentication() method
# - _setup_sp_auth() for OAuth M2M flow
# - Automatic token retrieval from service principal
```

#### Step 2: Re-log Model with Updated Code

**Run [Cell 15](#cell-a87cc582-c341-408f-9421-cc200725ac94)** to log the updated model.

```python
# This creates a new MLflow run with:
# - Updated WorkdaySalesRAG with OAuth support
# - requests library added to pip_requirements
# - Same config.json artifact
```

**Important:** Note the new `model_uri` from the output - you'll need it for registration.

#### Step 3: Register New Model Version (If Needed)

**Option A - First Deployment:**
Run [Cell 17](#cell-c4db0c68-1519-4cc9-a97c-971aaf0c3fa5) to register the model.

**Option B - Update Existing:**
Re-run Cell 17 to create a new version with the updated code.

#### Step 4: Update Endpoint with Service Principal Auth

**Run [Cell 19](#cell-42e08324-d810-44eb-9fbf-ebf966662038)** - Now configured with:
```python
auth_env_vars = {
    "DATABRICKS_HOST": "https://dbc-00781aa6-52b5.cloud.databricks.com",
    "DATABRICKS_CLIENT_ID": "{{secrets/agentic/sp_client_id}}",
    "DATABRICKS_CLIENT_SECRET": "{{secrets/agentic/sp_client_secret}}",
}
```

This will:
* Update the endpoint configuration
* Pass service principal credentials securely via secrets
* Trigger config update (~2-3 minutes)

#### Step 5: Wait for Deployment

**Run [Cell 21](#cell-e683d02a-7465-4cac-a6b8-3251c947ce9f)** to monitor endpoint readiness.

#### Step 6: Test!

**Run [Cell 22](#cell-e8933c0c-88c4-4424-ba2b-4e2cadede945)** to verify the endpoint works with authentication.

---

### 🔍 How OAuth Authentication Works

```mermaid
sequenceDiagram
    participant Endpoint as Model Serving<br/>Endpoint
    participant Model as WorkdaySalesRAG<br/>Model
    participant OAuth as OAuth Token<br/>Endpoint
    participant LLM as LLM Endpoint
    participant VS as Vector Search
    
    Endpoint->>Model: Invoke predict(question)
    Model->>Model: Read DATABRICKS_CLIENT_ID<br/>from environment
    Model->>OAuth: POST /oidc/v1/token<br/>(client_credentials grant)
    OAuth-->>Model: access_token
    Model->>Model: Set DATABRICKS_TOKEN=access_token
    Model->>VS: Query indexes with token
    VS-->>Model: Retrieved contexts
    Model->>LLM: Generate answer with token
    LLM-->>Model: Generated response
    Model-->>Endpoint: Return answer
```

**Key Points:**
* Service principal credentials are **never logged** - only passed as secret references
* OAuth token is obtained **at runtime** when the model loads
* Token is used for **all Databricks API calls** (LLM + Vector Search)
* Secrets are resolved by the serving infrastructure **before** passing to model

---

### 🆘 Troubleshooting

**❌ "OAuth token request failed: 401"**
* Check service principal credentials are correct
* Verify secrets in `agentic` scope: `dbutils.secrets.list('agentic')`

**❌ "Permission denied" on LLM/Vector Search**
* Grant `CAN_QUERY` on endpoints to `agentic_sp` (see checklist above)

**❌ "Service Principal authentication failed"**
* Check `DATABRICKS_HOST` format: must include `https://`
* Verify service principal is active (not disabled)

**❌ Model still returns authentication errors**
* Wait full 2-3 minutes for config update to complete
* Check endpoint state is `READY`
* Review endpoint logs for detailed error messages

---

### ✨ Next: Run Cells in Order

1. [Cell 14](#cell-1d8c1bba-2fcb-499b-b0b9-3d553c0fdadf) - Redefine model with OAuth
2. [Cell 15](#cell-a87cc582-c341-408f-9421-cc200725ac94) - Log updated model
3. [Cell 17](#cell-c4db0c68-1519-4cc9-a97c-971aaf0c3fa5) - Register (new version)
4. [Cell 19](#cell-42e08324-d810-44eb-9fbf-ebf966662038) - Update endpoint with SP auth
5. [Cell 21](#cell-e683d02a-7465-4cac-a6b8-3251c947ce9f) - Wait for readiness
6. [Cell 22](#cell-e8933c0c-88c4-4424-ba2b-4e2cadede945) - Test endpoint 🎉

In [0]:
# ============================================================================= 
# RUN THIS CELL AFTER YOU'VE SET UP THE SECRET
# =============================================================================
# Prerequisites:
# 1. Generated Personal Access Token
# 2. Stored token: databricks secrets put --scope rag_model_serving --key databricks_token
# =============================================================================

from databricks.sdk.service.serving import ServedEntityInput

# Authentication configuration
auth_env_vars = {
    "DATABRICKS_HOST": f"https://{w.config.host}",
    "DATABRICKS_TOKEN": "{{secrets/rag_model_serving/databricks_token}}"
}

print("🔧 Updating endpoint with authentication...\n")
print(f"Endpoint: {endpoint_name}")
print(f"Model: {registered_model_name} (v{model_version})")
print(f"Auth: Using secrets/rag_model_serving/databricks_token\n")

# Rebuild served entity with authentication
served_entity_with_auth = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=str(model_version),
    workload_type="CPU",
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero_enabled,
    environment_vars=auth_env_vars  # 👈 Authentication added here
)

try:
    # Update the endpoint configuration
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[served_entity_with_auth]
    )
    
    print("✅ Endpoint configuration updated successfully!\n")
    print("="*80)
    print("⏳ The endpoint is now updating (takes ~2-3 minutes)")
    print("="*80)
    print(f"\n🔗 Check status at:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    print("\n📝 What was updated:")
    print("   • Added DATABRICKS_HOST environment variable")
    print("   • Added DATABRICKS_TOKEN from secrets")
    print("   • Model can now authenticate to LLM and Vector Search endpoints")
    print("\n🧪 Next: Wait for update to complete, then re-run Cell 22 (Test Endpoint)")
    print("="*80)
    
except Exception as e:
    print(f"❌ Error updating endpoint: {str(e)}\n")
    print("👉 Common issues:")
    print("   1. Secret not found: Make sure you created the secret scope and stored the token")
    print("   2. Permission denied: Ensure you have permission to manage this endpoint")
    print("   3. Invalid token: Token may have expired or been revoked")
    print("\n🔍 Debug: Check if secret exists:")
    print("   Run: dbutils.secrets.list('rag_model_serving')")
    raise

In [0]:
# =============================================================================
# VERIFY SERVICE PRINCIPAL PERMISSIONS
# =============================================================================
# Run this BEFORE updating the endpoint to ensure your service principal
# has the necessary permissions to access LLM and Vector Search endpoints
# =============================================================================

import requests
import json

print("🔍 Verifying Service Principal Setup")
print("="*80)

# Check if secret exists
print("\n1️⃣ Checking if secret exists...")
try:
    secrets_list = dbutils.secrets.list('rag_model_serving')
    secret_keys = [s.key for s in secrets_list]
    
    if 'databricks_token' in secret_keys:
        print("   ✅ Secret found: rag_model_serving/databricks_token")
    else:
        print("   ❌ Secret NOT found: databricks_token")
        print("   👉 Create it: databricks secrets put --scope rag_model_serving --key databricks_token")
        raise ValueError("Secret not found")
except Exception as e:
    if "does not exist" in str(e).lower():
        print("   ❌ Secret scope NOT found: rag_model_serving")
        print("   👉 Create it: databricks secrets create-scope --scope rag_model_serving")
        raise
    else:
        print(f"   ⚠️  Error checking secrets: {e}")

# Instructions for granting permissions
print("\n2️⃣ Service Principal Permissions Required:")
print("-"*80)
print("\n🔑 Your Service Principal needs these permissions:\n")

print("🤖 LLM Endpoint Access:")
print("   Endpoint: databricks-meta-llama-3-3-70b-instruct")
print("   Permission: CAN_QUERY")
print("   How to grant:")
print("   1. Go to: Serving > Foundation Model APIs")
print("   2. Click on 'databricks-meta-llama-3-3-70b-instruct'")
print("   3. Click 'Permissions' tab")
print("   4. Add your Service Principal with 'CAN_QUERY' permission")

print("\n🔍 Vector Search Endpoint Access:")
print("   Endpoint: sales-endpoint-rag_agentic")
print("   Permission: CAN_QUERY")
print("   How to grant:")
print("   1. Go to: Compute > Vector Search")
print("   2. Click on 'sales-endpoint-rag_agentic'")
print("   3. Click 'Permissions' tab")
print("   4. Add your Service Principal with 'CAN_QUERY' permission")

print("\n📁 Vector Search Index Access:")
print("   Indexes:")
print("   • rag_agentic.workday_demos.customer_feedback_index")
print("   • rag_agentic.workday_demos.meeting_notes_index")
print("   • rag_agentic.workday_demos.email_communications_index")
print("   Permission: SELECT (read access on underlying Delta tables)")
print("   How to grant:")
print("   1. Go to: Catalog > rag_agentic > workday_demos")
print("   2. For each index's source table, grant SELECT permission")
print("   3. Or grant on the schema level for all tables")

print("\n" + "="*80)
print("✅ Checklist Before Running Next Cell:")
print("="*80)
print("\n☐ Secret created: rag_model_serving/databricks_token")
print("☐ Service Principal has CAN_QUERY on LLM endpoint")
print("☐ Service Principal has CAN_QUERY on Vector Search endpoint")
print("☐ Service Principal has SELECT on Vector Search indexes (or source tables)")
print("\nOnce all checked, proceed to next cell to update endpoint!")
print("="*80)

In [0]:
# =============================================================================
# UPDATE ENDPOINT WITH SERVICE PRINCIPAL AUTHENTICATION
# =============================================================================
# Prerequisites:
# 1. Service Principal created with appropriate permissions
# 2. Service Principal token generated
# 3. Token stored in Databricks Secrets:
#    databricks secrets put --scope rag_model_serving --key databricks_token
# =============================================================================

from databricks.sdk.service.serving import ServedEntityInput

print("🔐 Updating Endpoint with Service Principal Authentication")
print("="*80)

# Service Principal Authentication Configuration
# The secret reference {{secrets/scope/key}} is resolved at runtime by the endpoint
auth_env_vars = {
    "DATABRICKS_HOST": f"https://{w.config.host}",
    "DATABRICKS_TOKEN": "{{secrets/rag_model_serving/databricks_token}}",
}

print("📋 Configuration:")
print(f"   Endpoint: {endpoint_name}")
print(f"   Model: {registered_model_name} (v{model_version})")
print(f"   Auth Type: Service Principal")
print(f"   Secret Reference: {{secrets/rag_model_serving/databricks_token}}")
print(f"   Host: https://{w.config.host}\n")

# Rebuild served entity with service principal authentication
served_entity_with_sp_auth = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=str(model_version),
    workload_type="CPU",
    workload_size=workload_size,
    scale_to_zero_enabled=scale_to_zero_enabled,
    environment_vars=auth_env_vars  # 👈 Service Principal authentication
)

try:
    print("🔧 Updating endpoint configuration...")
    
    # Update the endpoint
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[served_entity_with_sp_auth]
    )
    
    print("\n✅ Endpoint configuration updated successfully!")
    print("="*80)
    print("⏳ Status: Endpoint is now updating (typically takes 2-3 minutes)")
    print("="*80)
    
    print("\n📝 What was configured:")
    print("   ✓ DATABRICKS_HOST environment variable")
    print("   ✓ DATABRICKS_TOKEN from secrets (Service Principal token)")
    print("   ✓ Model can now authenticate to:")
    print("     • LLM Endpoint: databricks-meta-llama-3-3-70b-instruct")
    print("     • Vector Search Endpoint: sales-endpoint-rag_agentic")
    print("     • Vector Search Indexes (3 indexes)")
    
    print(f"\n🔗 Monitor progress at:")
    print(f"   https://{w.config.host}/ml/endpoints/{endpoint_name}")
    
    print("\n🧪 Next Steps:")
    print("   1. Wait ~2-3 minutes for config update to complete")
    print("   2. Verify endpoint state shows 'READY'")
    print("   3. Re-run Cell 22 (Test Serving Endpoint)")
    print("   4. Endpoint should now return real answers!")
    print("="*80)
    
except Exception as e:
    print(f"\n❌ Error updating endpoint: {str(e)}")
    print("\n👉 Troubleshooting:")
    print("   1. Secret not found:")
    print("      → Verify: databricks secrets list --scope rag_model_serving")
    print("      → Should show: databricks_token")
    print("   2. Permission denied:")
    print("      → Ensure Service Principal has CAN_QUERY on LLM endpoint")
    print("      → Ensure Service Principal has CAN_QUERY on Vector Search endpoint")
    print("   3. Invalid token:")
    print("      → Regenerate service principal token")
    print("      → Re-store: databricks secrets put --scope rag_model_serving --key databricks_token")
    print("\n🔍 Check secret exists:")
    print("   Run in new cell: dbutils.secrets.list('rag_model_serving')")
    raise

## 🚀 Best Practices for Deploying to Higher Environments

### 1. Environment Strategy

**Separate Unity Catalog Namespaces:**
```
dev.workday_demos.workday_sales_rag          # Development
staging.workday_demos.workday_sales_rag      # Staging/UAT  
prod.workday_demos.workday_sales_rag         # Production
```

**Benefits:**
- Isolate environments completely
- Prevent accidental production changes
- Enable parallel development
- Clear promotion path

---

### 2. Model Versioning & Aliases

**Use Aliases Instead of "Latest":**
```python
# ✅ GOOD: Use aliases for mutable references
registry_client.set_registered_model_alias(
    name="prod.workday_demos.workday_sales_rag",
    alias="champion",      # Current production model
    version=model_version
)

# Deploy using alias (resolves to exact version internally)
models:/prod.workday_demos.workday_sales_rag@champion

# ❌ BAD: Using "latest" or implicit version
models:/prod.workday_demos.workday_sales_rag/latest
```

**Alias Strategy:**
- `champion` → Current production model
- `challenger` → A/B test candidate
- `staging` → Pre-production validation
- Pin deployments to **exact versions** resolved from aliases

---

### 3. Configuration Management

**Externalize All Configuration:**
```python
# Store in Unity Catalog volumes or secrets
CONFIG = {
    "dev": {
        "catalog": "dev",
        "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct",
        "vs_endpoint": "sales-endpoint-dev",
        "workload_size": "Small",
        "scale_to_zero": True
    },
    "prod": {
        "catalog": "prod",
        "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct",
        "vs_endpoint": "sales-endpoint-prod",
        "workload_size": "Medium",  # Higher capacity
        "scale_to_zero": False      # Always warm
    }
}
```

**Never hardcode:**
- Catalog/schema names
- Endpoint names
- Credentials
- Model versions

---

### 4. CI/CD with Databricks Asset Bundles (DABs)

**Structure:**
```yaml
# databricks.yml
bundle:
  name: workday_sales_rag

resources:
  jobs:
    rag_training_job:
      name: "Train Workday RAG - ${var.environment}"
      tasks:
        - task_key: train_and_register
          notebook_task:
            notebook_path: ./notebooks/train_rag.py
          
  model_serving_endpoints:
    workday_sales_rag:
      name: "workday_sales_rag_${var.environment}"
      config:
        served_entities:
          - entity_name: ${var.catalog}.workday_demos.workday_sales_rag
            entity_version: "${var.model_version}"
            workload_type: CPU
            workload_size: ${var.workload_size}

targets:
  dev:
    variables:
      environment: dev
      catalog: dev
      workload_size: Small
      
  prod:
    variables:
      environment: prod
      catalog: prod
      workload_size: Medium
```

**Deployment Commands:**
```bash
# Deploy to dev
databricks bundle deploy -t dev

# Deploy to prod
databricks bundle deploy -t prod
```

---

### 5. Testing & Validation

**Pre-Production Checks:**

1. **Model Validation:**
   ```python
   # Run MLflow evaluation on new model version
   eval_result = mlflow.genai.evaluate(
       data=staging_eval_data,
       predict_fn=predict_fn,
       scorers=[RelevanceToQuery(), Safety(), Guidelines(...)]
   )
   
   # Assert quality thresholds
   assert eval_result.metrics['relevance_to_query/mean'] >= 0.95
   assert eval_result.metrics['safety/mean'] == 1.0
   ```

2. **Integration Tests:**
   - Test with production-like data
   - Verify vector search connectivity
   - Check LLM endpoint availability
   - Validate response format

3. **Load Testing:**
   - Simulate concurrent requests
   - Measure p50, p95, p99 latency
   - Test scale-to-zero cold start time

---

### 6. Deployment Strategies

**Blue-Green Deployment:**
```python
# Deploy new version to separate endpoint
w.serving_endpoints.create(
    name="workday_sales_rag_green",
    config=new_config
)

# After validation, switch traffic
# (Update DNS or API Gateway routing)
```

**Canary Deployment (Traffic Splitting):**
```python
# Serve 95% from v1, 5% from v2
served_entities = [
    ServedEntityInput(
        entity_name="prod.workday_demos.workday_sales_rag",
        entity_version="1",
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Medium",
        scale_to_zero_enabled=False,
        traffic_percentage=95
    ),
    ServedEntityInput(
        entity_name="prod.workday_demos.workday_sales_rag",
        entity_version="2",
        workload_type=ServingModelWorkloadType.CPU,
        workload_size="Medium",
        scale_to_zero_enabled=False,
        traffic_percentage=5
    )
]
```

---

### 7. Monitoring & Observability

**Enable Inference Tables:**
```python
# Track all predictions for analysis
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayInferenceTableConfig,
)

ai_gateway_config = AiGatewayConfig(
    inference_table_config=AiGatewayInferenceTableConfig(
        catalog_name="prod",
        schema_name="monitoring",
        table_name_prefix="workday_rag",
        enabled=True
    )
)
```

**Monitor Key Metrics:**
- Request latency (p50, p95, p99)
- Error rate
- Token usage and costs
- Vector search performance
- Endpoint availability
- Cold start frequency

**Set Up Alerts:**
```sql
-- Example: Alert on high error rate
SELECT 
  COUNT(*) FILTER (WHERE status_code >= 400) / COUNT(*) as error_rate,
  window.start
FROM prod.monitoring.workday_rag_payload_table
WINDOW tumble(timestamp, INTERVAL 5 MINUTES)
HAVING error_rate > 0.05  -- Alert if >5% errors
```

---

### 8. Security & Compliance

**Access Control:**
```sql
-- Grant minimal permissions
GRANT EXECUTE ON MODEL prod.workday_demos.workday_sales_rag 
  TO `prod-rag-service-principal`;
  
GRANT SELECT ON TABLE prod.workday_demos.customer_feedback_index
  TO `prod-rag-service-principal`;
```

**Use Service Principals:**
- Never use personal credentials in production
- Create service principal per application
- Rotate credentials regularly
- Use managed identities when possible

**PII & Data Governance:**
- Tag sensitive data in Unity Catalog
- Apply column masking if needed
- Log predictions for audit trail
- Implement data retention policies

---

### 9. Rollback Strategy

**Quick Rollback:**
```python
# Instant rollback by changing alias
registry_client.set_registered_model_alias(
    name="prod.workday_demos.workday_sales_rag",
    alias="champion",
    version=previous_working_version  # Roll back to known-good version
)

# Then update endpoint to previous version
w.serving_endpoints.update_config(
    name=endpoint_name,
    served_entities=[previous_entity_config]
)
```

**Maintain Version History:**
- Keep last 3-5 production versions
- Document model lineage
- Tag versions with release notes

---

### 10. Documentation Requirements

**Model Card:**
- Model purpose and use cases
- Training data sources
- Performance metrics
- Known limitations
- Inference requirements
- Update history

**Deployment Runbook:**
- Pre-deployment checklist
- Deployment steps
- Validation procedures
- Rollback procedures
- Emergency contacts

---

### Quick Checklist for Production Deployment

- [ ] Model registered in prod Unity Catalog
- [ ] All configurations externalized
- [ ] Service principal credentials configured
- [ ] Integration tests passing
- [ ] Load tests completed
- [ ] Monitoring dashboards created
- [ ] Alerts configured
- [ ] Inference tables enabled
- [ ] Rollback plan documented
- [ ] Runbook updated
- [ ] Team notified
- [ ] Change request approved